# ML-Driven DCF Valuation Engine

**An institutional-grade pipeline that uses Machine Learning to forecast business drivers and compute the intrinsic value of publicly traded companies.**

```
FMP Financial Data
        ↓
Merge Financial Statements
        ↓
Feature Engineering
        ↓
Train ML Models
        ↓
Forecast Business Drivers
        ↓
Forecast Financial Statements
        ↓
DCF Valuation
        ↓
Intrinsic Value
        ↓
Compare vs Market Price
        ↓
Buy / Hold / Sell Signal
```

---
**Data Source:** Financial Modeling Prep (FMP) API — 14 Large-Cap US Equities, 5 Years of Annual Filings  
**Model:** Random Forest Regressor (one per business driver)  
**Valuation:** 5-Year Explicit Forecast + Gordon Growth Terminal Value  

---
## Step 1 — FMP Financial Data
Load the individual financial statement CSVs that were pulled from the FMP API.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# ── Load each financial statement from its own CSV ──
income     = pd.read_csv('fmp_dataset/csv/income-statement.csv')
balance    = pd.read_csv('fmp_dataset/csv/balance-sheet-statement.csv')
cashflow   = pd.read_csv('fmp_dataset/csv/cash-flow-statement.csv')
enterprise = pd.read_csv('fmp_dataset/csv/enterprise-values.csv')
quote      = pd.read_csv('fmp_dataset/csv/quote.csv')
dcf_fmp    = pd.read_csv('fmp_dataset/csv/discounted-cash-flow.csv')

print(f'Income Statement   : {income.shape}')
print(f'Balance Sheet      : {balance.shape}')
print(f'Cash Flow Statement: {cashflow.shape}')
print(f'Enterprise Values  : {enterprise.shape}')
print(f'Live Quotes        : {quote.shape}')
print(f'FMP DCF (benchmark): {dcf_fmp.shape}')
print(f'\nTickers: {sorted(income["symbol"].unique())}')

---
## Step 2 — Merge Financial Statements
Join Income Statement + Balance Sheet + Cash Flow + Enterprise Values on `(symbol, date)` to create one row per company-year with all the data we need.

In [ ]:
# ── Select only the columns we actually need from each statement ──
# This keeps the merge clean and avoids duplicate/ambiguous columns.

inc_cols = [
    'date', 'symbol', 'revenue', 'costOfRevenue', 'grossProfit',
    'researchAndDevelopmentExpenses', 'sellingGeneralAndAdministrativeExpenses',
    'operatingExpenses', 'operatingIncome', 'ebit', 'ebitda',
    'depreciationAndAmortization', 'interestExpense',
    'incomeBeforeTax', 'incomeTaxExpense', 'netIncome',
    'eps', 'epsDiluted', 'weightedAverageShsOut', 'weightedAverageShsOutDil'
]

bs_cols = [
    'date', 'symbol', 'cashAndCashEquivalents', 'shortTermInvestments',
    'netReceivables', 'inventory', 'totalCurrentAssets',
    'propertyPlantEquipmentNet', 'goodwillAndIntangibleAssets',
    'totalAssets', 'accountPayables', 'totalCurrentLiabilities',
    'longTermDebt', 'totalLiabilities', 'totalStockholdersEquity',
    'totalDebt', 'netDebt'
]

cf_cols = [
    'date', 'symbol', 'stockBasedCompensation', 'changeInWorkingCapital',
    'operatingCashFlow', 'capitalExpenditure', 'freeCashFlow',
    'acquisitionsNet', 'netCashProvidedByOperatingActivities',
    'commonStockRepurchased', 'commonDividendsPaid'
]

ev_cols = [
    'date', 'symbol', 'stockPrice', 'numberOfShares',
    'marketCapitalization', 'enterpriseValue'
]

# ── Merge on (symbol, date) using inner joins ──
merged = income[inc_cols].merge(balance[bs_cols],  on=['symbol', 'date'], how='inner')
merged = merged.merge(cashflow[cf_cols],           on=['symbol', 'date'], how='inner')
merged = merged.merge(enterprise[ev_cols],         on=['symbol', 'date'], how='inner')

# ── Sort chronologically within each company ──
merged['date'] = pd.to_datetime(merged['date'])
merged = merged.sort_values(['symbol', 'date']).reset_index(drop=True)

print(f'Merged dataset: {merged.shape[0]} rows x {merged.shape[1]} columns')
print(f'Companies: {merged["symbol"].nunique()}  |  Years per company: ~{merged.shape[0] // merged["symbol"].nunique()}')
merged.head(3)

---
## Step 3 — Feature Engineering
Calculate the **business drivers** that the DCF model needs. These are the ratios and growth rates that, when forecasted, let us reconstruct a full set of projected financial statements.

| Driver | Formula | Why It Matters |
|--------|---------|----------------|
| Revenue Growth | `(Rev_t / Rev_t-1) - 1` | Top-line trajectory |
| Gross Margin | `Gross Profit / Revenue` | Pricing power & COGS efficiency |
| EBIT Margin | `EBIT / Revenue` | Core operating profitability |
| Effective Tax Rate | `Tax / Pre-Tax Income` | Cash tax leakage |
| D&A % Revenue | `D&A / Revenue` | Asset intensity |
| Capex % Revenue | `|Capex| / Revenue` | Reinvestment rate |
| NWC % Revenue | `(Current Assets - Current Liabilities) / Revenue` | Working capital needs |

In [ ]:
def engineer_features(group):
    """Calculate business drivers for a single company's time-series."""
    g = group.copy()
    rev = g['revenue']

    # ── Growth ──
    g['rev_growth'] = rev.pct_change()

    # ── Margins ──
    g['gross_margin']  = g['grossProfit'] / rev
    g['ebit_margin']   = g['ebit'] / rev

    # ── Tax Rate (clipped to 0-40% to remove noise from loss years) ──
    g['eff_tax_rate'] = (g['incomeTaxExpense'] / g['incomeBeforeTax']).clip(0, 0.40)

    # ── Capital Intensity ──
    g['da_pct_rev']    = g['depreciationAndAmortization'] / rev
    g['capex_pct_rev'] = g['capitalExpenditure'].abs() / rev

    # ── Working Capital ──
    g['nwc']           = g['totalCurrentAssets'] - g['totalCurrentLiabilities']
    g['nwc_pct_rev']   = g['nwc'] / rev
    g['delta_nwc']     = g['nwc'].diff()

    # ── Lagged features (previous year's drivers → this year's input) ──
    driver_cols = ['rev_growth', 'gross_margin', 'ebit_margin',
                   'eff_tax_rate', 'da_pct_rev', 'capex_pct_rev', 'nwc_pct_rev']
    for col in driver_cols:
        g[f'{col}_lag1'] = g[col].shift(1)

    return g

merged = merged.groupby('symbol', group_keys=False).apply(engineer_features)

# ── Clean up infinities and NaNs from pct_change / division ──
merged = merged.replace([np.inf, -np.inf], np.nan)

print('Feature engineering complete.')
print(f'Columns added: rev_growth, gross_margin, ebit_margin, eff_tax_rate, da_pct_rev, capex_pct_rev, nwc_pct_rev + lags')

# Show a snapshot of the drivers for one company
driver_preview = ['date', 'symbol', 'revenue', 'rev_growth', 'ebit_margin',
                  'eff_tax_rate', 'da_pct_rev', 'capex_pct_rev', 'nwc_pct_rev']
merged[merged['symbol'] == 'AAPL'][driver_preview]

---
## Step 4 — Train ML Models
We train one **Random Forest Regressor** per business driver. The model learns cross-sectional patterns across all 14 companies (e.g., "companies with high gross margins tend to maintain them").

**Why Random Forest?**
- Handles non-linear relationships between financial ratios.
- Robust to outliers (common in financial data).
- Built-in feature importance for interpretability.
- No feature scaling required.

In [ ]:
# ── Define features (lagged drivers) and targets (current drivers) ──
FEATURES = [
    'rev_growth_lag1', 'gross_margin_lag1', 'ebit_margin_lag1',
    'eff_tax_rate_lag1', 'da_pct_rev_lag1', 'capex_pct_rev_lag1', 'nwc_pct_rev_lag1'
]

TARGETS = [
    'rev_growth', 'gross_margin', 'ebit_margin',
    'eff_tax_rate', 'da_pct_rev', 'capex_pct_rev', 'nwc_pct_rev'
]

# ── Training data: rows where all features AND targets exist ──
train_df = merged.dropna(subset=FEATURES + TARGETS).copy()
print(f'Training samples: {len(train_df)} (from {train_df["symbol"].nunique()} companies)')

# ── Train one model per target driver ──
models = {}
model_scores = {}

for target in TARGETS:
    X = train_df[FEATURES]
    y = train_df[target]

    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=4,
        min_samples_leaf=3,
        random_state=42
    )
    rf.fit(X, y)
    models[target] = rf

    # Cross-validated R² score for reporting
    cv_scores = cross_val_score(rf, X, y, cv=min(3, len(train_df)), scoring='r2')
    model_scores[target] = cv_scores.mean()

# ── Display model performance ──
score_df = pd.DataFrame({
    'Business Driver': TARGETS,
    'Cross-Val R²': [f'{model_scores[t]:.3f}' for t in TARGETS]
})
print('\n── Model Performance (Cross-Validated R²) ──')
print(score_df.to_string(index=False))

---
## Steps 5 & 6 — Forecast Business Drivers → Forecast Financial Statements
For each company, we take the most recent year's drivers as input, then **iteratively forecast 5 years forward**. Each year's predictions become the next year's lagged features.

From the forecasted drivers, we reconstruct the line items needed for Free Cash Flow to Firm (FCFF):

```
FCFF = NOPAT + D&A − Capex − ΔNWC
     = EBIT × (1 − Tax Rate) + D&A − Capex − ΔNWC
```

In [ ]:
# ── DCF Assumptions ──
WACC                = 0.09    # 9% weighted average cost of capital
TERMINAL_GROWTH     = 0.025   # 2.5% perpetual growth (long-run GDP)
FORECAST_YEARS      = 5

# ── Get the latest fiscal year for each company ──
latest = merged.sort_values('date').groupby('symbol').tail(1).copy()
latest = latest.dropna(subset=FEATURES)  # need valid features to start forecasting

print(f'Running DCF for {len(latest)} companies...\n')

all_results = []
all_projections = {}  # store detailed projections per company

for _, row in latest.iterrows():
    sym = row['symbol']

    # ── Starting state ──
    curr_rev = row['revenue']
    curr_nwc = row['nwc']

    # Build the feature vector from the latest year's actual drivers
    feat_dict = {f: row[f] for f in FEATURES}
    feat_input = pd.DataFrame([feat_dict])

    yearly_detail = []
    fcff_list = []

    for yr in range(1, FORECAST_YEARS + 1):
        # ── Step 5: Predict next year's drivers ──
        preds = {t: models[t].predict(feat_input)[0] for t in TARGETS}

        # ── Step 6: Reconstruct financial statement line items ──
        proj_rev   = curr_rev * (1 + preds['rev_growth'])
        proj_gross = proj_rev * preds['gross_margin']
        proj_ebit  = proj_rev * preds['ebit_margin']
        proj_tax   = proj_ebit * preds['eff_tax_rate']
        proj_nopat = proj_ebit - proj_tax
        proj_da    = proj_rev * preds['da_pct_rev']
        proj_capex = proj_rev * preds['capex_pct_rev']
        proj_nwc   = proj_rev * preds['nwc_pct_rev']
        d_nwc      = proj_nwc - curr_nwc

        # ── FCFF = NOPAT + D&A - Capex - ΔNWC ──
        fcff = proj_nopat + proj_da - proj_capex - d_nwc
        fcff_list.append(fcff)

        yearly_detail.append({
            'Year': yr,
            'Revenue': proj_rev,
            'Rev Growth': preds['rev_growth'],
            'EBIT Margin': preds['ebit_margin'],
            'NOPAT': proj_nopat,
            'D&A': proj_da,
            'Capex': proj_capex,
            'Delta NWC': d_nwc,
            'FCFF': fcff
        })

        # ── Roll forward: this year's predictions become next year's lags ──
        feat_dict = {f'{t}_lag1': preds[t] for t in TARGETS}
        feat_input = pd.DataFrame([feat_dict])
        curr_rev = proj_rev
        curr_nwc = proj_nwc

    all_projections[sym] = pd.DataFrame(yearly_detail)

    # ── Step 7: DCF Valuation ──
    discount_factors = [(1 + WACC) ** yr for yr in range(1, FORECAST_YEARS + 1)]
    pv_fcff = sum(f / d for f, d in zip(fcff_list, discount_factors))

    terminal_value    = (fcff_list[-1] * (1 + TERMINAL_GROWTH)) / (WACC - TERMINAL_GROWTH)
    pv_terminal_value = terminal_value / discount_factors[-1]

    enterprise_value  = pv_fcff + pv_terminal_value

    # ── Step 8: Intrinsic Value per Share ──
    cash       = row['cashAndCashEquivalents']
    debt       = row['totalDebt']
    shares     = row['weightedAverageShsOutDil']

    equity_value = enterprise_value + cash - debt
    intrinsic_per_share = equity_value / shares if shares > 0 else np.nan

    all_results.append({
        'Symbol': sym,
        'Last FY Revenue': row['revenue'],
        'PV of FCFFs': pv_fcff,
        'PV of Terminal Value': pv_terminal_value,
        'Enterprise Value': enterprise_value,
        '(+) Cash': cash,
        '(-) Debt': debt,
        'Equity Value': equity_value,
        'Shares Outstanding': shares,
        'ML Intrinsic Value': intrinsic_per_share
    })

valuation = pd.DataFrame(all_results)
print('\u2714 Forecasting & DCF complete for all companies.')

### Projected Financial Statements (Example: AAPL)
Show the year-by-year ML-forecasted financials for Apple.

In [ ]:
# ── Display the 5-year projection for one company ──
example_sym = 'AAPL'
proj = all_projections[example_sym].copy()

# Format large numbers in billions for readability
money_cols = ['Revenue', 'NOPAT', 'D&A', 'Capex', 'Delta NWC', 'FCFF']
for col in money_cols:
    proj[col] = proj[col].apply(lambda x: f'${x/1e9:.1f}B')
proj['Rev Growth']  = proj['Rev Growth'].apply(lambda x: f'{x:.1%}')
proj['EBIT Margin'] = proj['EBIT Margin'].apply(lambda x: f'{x:.1%}')

print(f'\n── 5-Year Projected Financials: {example_sym} ──')
proj

---
## Steps 9 & 10 — Compare vs Market Price → Buy / Hold / Sell Signal
Merge with live quote prices, compute the margin of safety, and generate a trading signal.

| Signal | Condition |
|--------|-----------|
| **BUY** | Intrinsic Value > 20% above Market Price (undervalued) |
| **HOLD** | Within ±20% band |
| **SELL** | Intrinsic Value > 20% below Market Price (overvalued) |

In [ ]:
# ── Bring in live market prices from the quote CSV ──
live_prices = quote[['symbol', 'name', 'price']].rename(
    columns={'symbol': 'Symbol', 'price': 'Market Price', 'name': 'Company'}
)

# ── Also bring in FMP's own DCF for comparison ──
fmp_dcf = dcf_fmp[['symbol', 'dcf']].rename(
    columns={'symbol': 'Symbol', 'dcf': 'FMP DCF Value'}
)

# ── Merge everything ──
final = valuation[['Symbol', 'ML Intrinsic Value']].merge(live_prices, on='Symbol', how='inner')
final = final.merge(fmp_dcf, on='Symbol', how='left')

# ── Compute upside / downside ──
final['ML Upside %'] = ((final['ML Intrinsic Value'] - final['Market Price']) / final['Market Price']) * 100

# ── Generate Signal ──
def signal(upside):
    if pd.isna(upside):
        return 'N/A'
    if upside > 20:
        return '🟢 BUY'
    elif upside < -20:
        return '🔴 SELL'
    else:
        return '🟡 HOLD'

final['Signal'] = final['ML Upside %'].apply(signal)

# ── Format for display ──
display_df = final[['Symbol', 'Company', 'Market Price',
                     'ML Intrinsic Value', 'FMP DCF Value',
                     'ML Upside %', 'Signal']].copy()

display_df['Market Price']       = display_df['Market Price'].apply(lambda x: f'${x:,.2f}')
display_df['ML Intrinsic Value'] = display_df['ML Intrinsic Value'].apply(lambda x: f'${x:,.2f}')
display_df['FMP DCF Value']      = display_df['FMP DCF Value'].apply(lambda x: f'${x:,.2f}')
display_df['ML Upside %']        = display_df['ML Upside %'].apply(lambda x: f'{x:+.1f}%')

display_df = display_df.sort_values('Signal')
print('\n═══  FINAL VALUATION REPORT  ═══\n')
display_df

---
## Appendix — DCF Bridge (Detailed Breakdown)
Show the full Enterprise Value → Equity Value waterfall for every company.

In [ ]:
# ── Full DCF waterfall for each company ──
bridge = valuation.copy()
money_bridge_cols = ['Last FY Revenue', 'PV of FCFFs', 'PV of Terminal Value',
                     'Enterprise Value', '(+) Cash', '(-) Debt', 'Equity Value']

for col in money_bridge_cols:
    bridge[col] = bridge[col].apply(lambda x: f'${x/1e9:,.1f}B')

bridge['Shares Outstanding'] = bridge['Shares Outstanding'].apply(lambda x: f'{x/1e9:,.2f}B')
bridge['ML Intrinsic Value'] = bridge['ML Intrinsic Value'].apply(lambda x: f'${x:,.2f}')

print('\n── DCF Bridge: Enterprise Value → Intrinsic Value per Share ──\n')
bridge